In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,457.69",-1.01%,-1.14%,-1.55%,+0.51%,+0.16%,+0.16%
NASDAQ Composite,^IXIC,"25,520.24",-1.40%,-2.25%,-2.90%,-1.93%,-2.94%,-2.94%
Apple,AAPL,333.74,+0.14%,+6.00%,+5.84%,+12.77%,+9.43%,+9.43%
NVIDIA,NVDA,202.81,-2.21%,-4.24%,-3.86%,-0.90%,-7.50%,-7.50%
Microsoft,MSFT,393.82,-1.82%,+2.31%,+2.26%,+3.93%,-6.03%,-6.03%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"64,141.12",-4.03%,-5.32%,-6.44%,-9.98%,+3.98%,+3.98%
Toyota Motor,7203.T,"2,899.50",-0.33%,+2.13%,+2.71%,+4.43%,-2.64%,-2.64%
Sony Group,6758.T,"3,470.00",+0.93%,+2.57%,+3.30%,+10.51%,-2.36%,-2.36%
Mitsubishi UFJ Financial,8306.T,"3,473.00",-4.30%,-3.53%,+0.35%,+5.95%,+13.24%,+13.24%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,509.43",-0.54%,+0.25%,+0.73%,+6.10%,+9.19%,+9.19%
DBS Group,D05.SI,71.96,-0.72%,-0.10%,+2.14%,+9.10%,+16.53%,+16.53%
UOB,U11.SI,42.47,-2.37%,-4.30%,-4.30%,+8.20%,+12.68%,+12.68%
Singtel,Z74.SI,4.44,+1.14%,+0.00%,+0.91%,+2.07%,-5.53%,-5.53%
CapitaLand Ascendas REIT,A17U.SI,2.51,+1.21%,+0.80%,-0.40%,+0.00%,+0.00%,+0.00%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,251.00",-2.93%,-1.81%,-2.25%,-3.71%,+1.41%,+1.41%
ニデック (6594),6594.T,"2,482.00",-3.91%,-4.83%,-6.69%,-7.90%,-5.63%,-5.63%
未来工業 (7931),7931.T,"3,210.00",-1.83%,-0.93%,-0.16%,+6.64%,+5.94%,+5.94%
東部ネットワーク (9036),9036.T,"1,360.00",+2.64%,+10.12%,+7.51%,+10.03%,-2.16%,-2.16%
ニトリHD (9843),9843.T,"2,459.00",+1.84%,+0.27%,+3.32%,+4.28%,-0.61%,-0.61%
MRK HD (9980),9980.T,93.00,-2.11%,-1.06%,-3.12%,-1.06%,-3.12%,-3.12%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,98.20,+0.07%,+0.20%,+0.12%,-0.08%,+0.53%,+0.53%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.94,-1.05%,+0.00%,-0.53%,+1.08%,-1.57%,-1.57%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.84,+2.16%,+0.35%,+5.19%,+2.90%,-1.39%,-1.39%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,-1.85%,-1.85%,-1.85%,-5.36%,-5.36%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.16,+0.48%,+0.73%,+1.22%,+5.58%,+3.48%,+3.48%
First REIT (ファースト・リート),AW9U.SI,0.22,+0.00%,+0.00%,-2.17%,-2.17%,-4.26%,-4.26%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.54,+0.00%,-0.92%,-2.70%,-6.09%,-10.74%,-10.74%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+0.00%,+0.00%,-0.63%,+2.60%,+2.60%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,+0.00%,-2.22%,-4.35%,-2.22%,-12.00%,-12.00%
Medtecs International (医療用消耗品),546.SI,0.11,+0.89%,+0.00%,-0.88%,-5.04%,-13.08%,-13.08%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,-1.41%,+0.00%,+0.00%,+1.45%,+9.38%,+9.38%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.87,-0.77%,-0.51%,+0.52%,+3.75%,+7.20%,+7.20%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.21,+0.83%,-0.82%,-1.63%,-5.47%,+0.83%,+0.83%
Riverstone,AP4.SI,0.86,-1.16%,+0.00%,+0.00%,+1.79%,-8.56%,-8.56%
ヤンジジャン・シップビルディング(BS6),BS6.SI,3.63,-0.55%,+1.68%,+0.55%,-1.63%,-1.89%,-1.89%
Sembcorp Ind (U96),U96.SI,5.36,-2.19%,-2.72%,-5.63%,-15.19%,-13.13%,-13.13%


<div style="background-color: #EBCB8B; color: #2E3440; padding: 12px; border-radius: 6px; font-weight: bold;">
</div>